# Lesson 94 · Six months boiled down to five minutes — the Aurora v1 demo and assembling the evidence

**Goal**: polish a demo you can show — an end-to-end storyline + an acceptance checklist, so that "I really built it, and I really understand it" holds up under questioning.

🔗 **Aurora connection**:
- `src/aurora/audio/` — Audio Core (FFT / STFT / Mel / MFCC, all written by hand)
- `src/aurora/mlops/`  — training loop and model weights
- `src/aurora/speech/` — speech recognition module
- `src/aurora/music/`  — music features and recommendation (features.py / similarity.py)
- `src/aurora/llm/`    — LLM integration layer (kvcache / retrieve / sample)
- `src/aurora/rag/`    — retrieval-augmented generation
- `tests/`            — benchmark tests for every module (checked against numpy / reference formulas)


← **Previous lesson** · [L93 · MLOps basics](L93_mlops.ipynb)

> Last lesson covered **MLOps basics**: experiment tracking with W&B, model versioning, Docker packaging and deployment scripts.
> This lesson explores **the Aurora v1 demo**.

## Learning goals

After this lesson you should be able to:

1. **Use the WWHRL framework** to give a structured five-minute introduction to Aurora, covering What / Why / How / Results / Limits.
2. **Show the tests passing with pytest**, using the code benchmarks (error < 1e-10) as verifiable pieces of evidence.
3. **Fill in a complete evidence checklist** (four layers: GitHub history / blog / tests / screen recording), with at least one showable artifact per layer.
4. **Write 6 achievement bullets that make each milestone clear** (method + capability + checkable metric), one for each of the milestones M1–M6.


## Why you need a "delivery mindset"

The value of a research system depends not only on what it can do, but on whether someone else can grasp what it does within five minutes.
Aurora v1's core selling point is that it's **built from scratch** (no API wrappers): when someone asks "how do you do the FFT?" (an interviewer, a colleague, or perhaps your future self), the answer isn't "with librosa" —
it's opening a notebook and running it right there. This lesson organizes six months of work into a clear **evidence chain**,
and pins down the demo flow, the common questions and the achievement bullets.


In [ ]:
import numpy as np


## 1. The evidence chain

"I implemented the FFT from scratch" — that sentence needs **verifiable** evidence behind it; otherwise nobody (not even you) can tell whether you truly understand it or just memorized interview answers.
Aurora's evidence chain has four layers:

| Layer | Form | What it proves |
|------|------|------|
| **GitHub history** | commit log + PRs | proves the process, not just the result |
| **Blog posts** | `docs/current/blog/` Markdown | proves depth of understanding — you can explain it |
| **Test results** | `pytest tests/` passing | proves the code is correct, checked against formulas |
| **Demo recording** | GIF / MP4 | proves the system runs end to end |

The core logic of each layer: **the further down, the harder to fake**. Commit timestamps can't be changed, the tests hold formulas to an error bound,
and a screen recording shows the real inference latency. Prepare your work along these four layers, and fill in whichever one is missing.


In [ ]:
# Evidence-chain self-check: print the key file path for each layer and confirm it exists
import os

AURORA_ROOT = os.path.expanduser("~/AURORA")

evidence = {
    "GitHub history": ".git/",
    "Blog posts":     "docs/current/blog/",
    "Tests":          "tests/",
    "ADR records":    "docs/current/adr/",
}

print("── Evidence-chain self-check ──")
for label, rel in evidence.items():
    full = os.path.join(AURORA_ROOT, rel)
    exists = os.path.exists(full)
    status = "✅" if exists else "❌ missing"
    print(f"  {status}  {label:12s}  {full}")


## 2. How to present an AI system: the WWHRL framework

When you introduce a system to someone in five minutes (an interviewer, a colleague, or your future self), they are trying to judge: can this person drive an AI system forward on their own?
A structured account works far better than a pile of details. We recommend **WWHRL** (What / Why / How / Results / Limits):

```
What    — what the system does, in one sentence
Why     — why build it from scratch instead of using an API (research integrity + real understanding)
How     — the key algorithms, which you can unpack (FFT → STFT → Mel → MFCC → model)
Results — quantifiable metrics (WER, latency, test pass rate)
Limits  — an honest account of the boundaries (single speaker, small vocabulary, not real time)
```

**The trap**: spending most of the time on What and skipping Why and Limits. Listeners care most about Why (it shows depth of thinking)
and Limits (it shows self-awareness). Results give a credibility anchor, but Results without a Why are hollow.

In [ ]:
# WWHRL template: print the standard Aurora v1 introduction (memorize it as is, or rewrite it)
wwhr = {
    "What": (
        "Aurora is an audio AI system built from scratch, "
        "covering four modules: feature extraction (MFCC), speech recognition (ASR), recommendation and an agent."
    ),
    "Why": (
        "Every core algorithm (FFT, mel filterbank, DCT) is written by hand, with no calls to librosa / scipy.signal. "
        "The aim is to demonstrate a first-principles understanding of DSP maths, not the ability to call an API."
    ),
    "How": (
        "The FFT uses the Cooley-Tukey butterfly decomposition, O(N log N); "
        "the mel filterbank builds its triangular filters by hand from the HTK formula; "
        "MFCC applies a hand-written DCT-II to the mel energies; "
        "the training loop implements gradient descent in pure NumPy (or lightweight PyTorch)."
    ),
    "Results": (
        "Every module passes error tests against numpy.fft / reference formulas (< 1e-10); "
        "the end-to-end ASR demo on the LibriSpeech clean subset [to fill in: record the WER after running scripts/eval_asr.py]."
    ),
    "Limits": (
        "Currently supports only a single speaker at 16kHz sampling; "
        "inference is not real time (no streaming decoding); "
        "the model hasn't been tuned on large-scale data, so its WER is higher than industrial systems."
    ),
}

print("── Aurora v1 WWHRL introduction ──\n")
for key, val in wwhr.items():
    print(f"[{key}]\n  {val}\n")

## 3. Résumé excerpts: one bullet per month

The format for an AI-system bullet on a résumé:

```
[method] implemented [capability], [metric / comparison]
```

Examples:

```
• Hand-wrote a Cooley-Tukey FFT (O(N log N)), error < 1e-10 (vs numpy.fft)
• Built an 80-channel mel filterbank + DCT-II: an end-to-end MFCC pipeline with no third-party DSP library
• Trained a lightweight ASR model, LibriSpeech clean WER < X%, inference latency < Y ms (CPU)
• Implemented an audio recommendation engine based on MFCC similarity, top-5 precision Z%
• Built an agent layer that orchestrates the feature extraction → inference → result aggregation pipeline
```

Tips for writing each bullet:
- **Precise verbs**: "hand-wrote", "built", "implemented", "assembled" — not "used" or "based on"
- **Checkable metrics**: error bounds come from the test logs, WER from the evaluation script
- **A comparison anchor**: a comparison (vs numpy.fft, vs librosa) is what makes it convincing


In [ ]:
# Résumé bullet generator: print the bullets completed so far, month by month
bullets = [
    ("M1", "Audio Core",  "Hand-wrote a Cooley-Tukey FFT (O(N log N)), numerical error < 1e-10 (vs numpy.fft)"),
    ("M2", "STFT / Mel",  "Implemented STFT + an 80-channel mel filterbank (HTK formula), no librosa / scipy.signal"),
    ("M3", "MFCC",        "Built a hand-written DCT-II; the full MFCC pipeline passes the formula benchmark tests"),
    ("M4", "Training loop",    "Built a training loop from scratch; the loss curve converges and the model weights are reproducible"),
    ("M5", "ASR",         "End-to-end speech recognition module, audio in → text out, [to fill in: record the WER after running scripts/eval_asr.py]"),
    ("M6", "Agent/Demo",  "Agent orchestration layer + v1 demo, covering the full feature extraction → inference → recommendation chain"),
]

print("── Résumé bullets (by month) ──\n")
for month, module, text in bullets:
    print(f"  [{month}] {module}")
    print(f"    • {text}")
    print()

# Check: the number of bullets should equal the number of milestones
assert len(bullets) == 6, f"expected 6 bullets, got {len(bullets)}"
print("✅ All 6 bullets are filled in")

## 4. The HuggingFace ecosystem and where Aurora stands

Aurora's "no API wrappers" principle applies to the **core DSP algorithms** (FFT / Mel / MFCC).
For large models beyond what you can write by hand, **knowing and using HuggingFace** is an essential skill for a research engineer.

### Core patterns in HuggingFace `transformers`

```python
from transformers import pipeline, AutoModel, AutoTokenizer

# 1. pipeline: the fastest way in, inference in one line
asr = pipeline("automatic-speech-recognition", model="openai/whisper-small")
result = asr("audio.wav")  # → {"text": "..."}

# 2. AutoModel: when you need hidden states or embeddings
model = AutoModel.from_pretrained("laion/larger_clap_general")
tokenizer = AutoTokenizer.from_pretrained("laion/larger_clap_general")

# 3. from_pretrained arguments control precision and device
model = AutoModel.from_pretrained(
    "facebook/musicgen-small",
    torch_dtype=torch.float16,   # half precision saves GPU memory
    device_map="auto",           # place on CPU/GPU automatically
)
```

### How Aurora and HuggingFace split the work

| Module | Aurora implements it itself | Used through HuggingFace |
|------|----------------|----------------------|
| Feature extraction | FFT / STFT / Mel / MFCC (from scratch) | — |
| KWS | simple CNN + trained ourselves | — |
| ASR | CTC forward algorithm (from scratch) | Whisper inference / fine-tuning |
| Music embedding | MusicEncoder + Triplet/NT-Xent | CLAP (large-scale version) |
| Music generation | understanding how EnCodec works | MusicGen (called directly) |
| LLM / RAG | TF-IDF / hand-written Transformer (principles) | Claude / GPT API (applications) |

**The key judgment**: being able to say "I implemented X because I wanted to truly understand it; I use HuggingFace's Y because implementing Y from scratch would take a year and 100,000 GPU hours" — that is the judgment of a mature builder, rather than "write everything myself" or "import everything".

In [ ]:
import numpy as np
import os

# ── Aurora capability self-check (an executable check instead of a static table) ──────────────────────────────
# Note the criterion: this matrix marks ✅/🔲 by "has it been distilled into src/?" (whether the matching file exists).
# The CTC forward algorithm and the asyncio pipeline have already been hand-written and verified in the L68-69 / L92 notebooks,
# they just haven't been moved into src/ yet, so they still show 🔲 to do here.
AURORA_ROOT = os.path.expanduser("~/AURORA")

capabilities = [
    # (capability, how it's proven, module path)
    ("FFT O(N log N)",          "error < 1e-10 vs numpy.fft",    "src/aurora/audio/transforms.py"),
    ("STFT windowing/framing",           "shape, magnitude match scipy",       "src/aurora/audio/stft.py"),
    ("Mel filterbank (80ch)",     "HTK formula, energy matches librosa",    "src/aurora/audio/mel.py"),
    ("Hand-written DCT-II",             "error < 1e-12 vs scipy.fft.dct", "src/aurora/audio/transforms.py"),
    ("Full MFCC pipeline",         "matches librosa.feature.mfcc",  "src/aurora/audio/mfcc.py"),
    ("KWS CNN classifier",          "planned — not implemented yet",              "src/aurora/mlops/kws.py (planned)"),
    ("CTC forward algorithm",            "path probabilities checked by hand",              "src/aurora/speech/ctc.py"),
    ("Levenshtein / WER",       "exact match with jiwer",             "src/aurora/speech/metrics.py"),
    ("MusicEncoder NT-Xent",    "t-SNE separates 3 clusters",            "src/aurora/music/embed.py"),
    ("Cosine k-NN recommendation",          "Precision@5 > 0.5",             "src/aurora/music/similarity.py"),
    ("TF-IDF RAG",              "no faiss / sentence-transformers","src/aurora/llm/retrieve.py"),
    ("asyncio pipeline",          "parallel ASR+RAG, p95 < 3s",        "src/aurora/realtime/runner.py"),
]

print("── Aurora v1 capability matrix ──\n")
print(f"{'Capability':30s}  {'How it is proven':35s}  {'Status'}")
print("─" * 85)
for cap, proof, path in capabilities:
    full = os.path.join(AURORA_ROOT, path)
    exists = os.path.exists(full)
    status = "✅" if exists else "🔲 to do"
    print(f"{cap:30s}  {proof:35s}  {status}")

print()
done = sum(1 for _, _, p in capabilities if os.path.exists(os.path.join(AURORA_ROOT, p)))
print(f"Implemented {done}/{len(capabilities)} core capabilities")
print("(Criterion: distilled into src/. The CTC / asyncio pipeline were hand-written and verified in L68-69 / L92, but not yet moved into src/)")

# ── HuggingFace model cheat sheet (the external large models Aurora relies on) ────────────────────────────
print("\n── HuggingFace models Aurora relies on ──\n")
hf_models = [
    ("openai/whisper-small",          "ASR inference / fine-tune","~460MB", "transformers.pipeline"),
    ("openai/whisper-tiny",           "ASR quick prototyping",    "~150MB", "transformers.pipeline"),
    ("laion/larger_clap_general",     "audio-text contrastive emb.", "~900MB", "ClapModel"),
    ("facebook/musicgen-small",       "text → music generation",  "~1.5GB", "MusicgenForConditionalGeneration"),
    ("facebook/encodec_32khz",        "audio tokenization",       "~100MB", "EncodecModel"),
    ("google/flan-t5-small",          "light text encoder (RAG)", "~300MB", "T5EncoderModel"),
]
print(f"{'Model ID':42s}  {'Use':25s}  {'Size':8s}  {'HuggingFace class'}")
print("─" * 100)
for model_id, use, size, cls in hf_models:
    print(f"{model_id:42s}  {use:25s}  {size:8s}  {cls}")

print("\n# Generic loading template:")
print("from transformers import AutoModel, AutoProcessor")
print('model = AutoModel.from_pretrained("<model_id>", device_map="auto")')

## Parameter experiment: a five-minute demo rehearsal + common Q&A

### The five-minute demo flow

| Time | Content | Key action |
|------|------|---------|
| 0:00–0:30 | Introduce Aurora in one sentence | Give the What + Why from WWHRL |
| 0:30–1:30 | Run the FFT notebook  | Run `notebooks/5_audio_dsp/L42_visual_fft.ipynb` (🎨 the FFT illustrated), show the butterfly diagram |
| 1:30–2:30 | Run the MFCC pipeline   | Run `notebooks/5_audio_dsp/L53_visual_mfcc.ipynb` (🎨 MFCC illustrated), show the MFCC heatmap |
| 2:30–3:30 | Run the ASR demo      | Run `notebooks/7_asr/L75_visual_asr.ipynb` (🎨 the ASR pipeline illustrated), show waveform → tokens → text |
| 3:30–4:30 | Show the test results       | `pytest tests/ -v`, all green |
| 4:30–5:00 | Limits + Q&A       | Raise the limitations yourself and invite follow-up questions |

**Prepare 10 common questions** (people will ask them, and you should be able to answer them yourself; answer outlines are in the code cell below):
1. Why not use librosa?
2. What is the time complexity of the FFT, and how did you verify it?
3. Why aren't the mel filterbank's frequencies spaced linearly?
4. What's the difference between the DCT-II and the DFT?
5. What does the 0th MFCC coefficient represent?
6. How does your model do in noisy conditions?
7. How would you extend it to streaming (real-time) inference?
8. What similarity measure does the recommendation module use?
9. How does the agent layer handle failures and retries?
10. If you had three months, what would you improve first?

In [ ]:
# Print the common Q&A: run it once and make sure every answer is in your head
qa = [
    (
        "Why not use librosa?",
        "librosa.stft / librosa.feature.mfcc are black boxes — you can't explain what happens inside. "
        "Writing them by hand forces you to understand every step of the maths; when pressed, you can go deeper instead of just saying 'I called a function'."
    ),
    (
        "What is the time complexity of the FFT, and how did you verify it?",
        "O(N log N): Cooley-Tukey does N/2 butterflies per stage, over log₂N stages. "
        "How to verify: time it for different N; the slope on a log-log plot should be about 1 (plus the log N factor), "
        "and compare with numpy.fft for a numerical error < 1e-10."
    ),
    (
        "Why aren't the mel filterbank's frequencies spaced linearly?",
        "The ear resolves low frequencies finely and high frequencies coarsely, and the mel scale mimics this. "
        "With the formula `mel = 2595 * log10(1 + hz/700)`, equal mel steps span small Hz gaps at low frequencies, "
        "so the filters are dense at low frequencies and sparse at high ones."
    ),
    (
        "What's the difference between the DCT-II and the DFT?",
        "The DFT assumes the signal repeats periodically, which creates a discontinuity at the boundary; the DCT-II assumes an even-symmetric extension, "
        "so the boundary is continuous and the energy concentrates in the low coefficients — good for compression and feature extraction. "
        "MFCC uses the DCT-II because the mel energies are highly correlated; after the DCT decorrelates them, the coefficients are more independent."
    ),
    (
        "What does the 0th MFCC coefficient represent?",
        "C0 is a weighted sum of the energies in all the mel bands, roughly the log total energy (loudness). "
        "In practice it's often dropped or replaced by the log frame energy, because it's sensitive to channel differences."
    ),
    (
        "How does your model do in noisy conditions?",
        "There is no noise augmentation yet, so WER rises sharply at SNR < 10dB. "
        "Ways to improve: data augmentation (added noise / reverb) or front-end spectral subtraction. This is a known limit."
    ),
    (
        "How would you extend it to streaming (real-time) inference?",
        "The STFT has to become a sliding window that processes each frame incrementally; the model switches to streaming decoding (CTC greedy or beam). "
        "The current batch inference framework doesn't support this; the inference entry point would need restructuring."
    ),
    (
        "What similarity measure does the recommendation module use?",
        "Cosine similarity between MFCC vectors. Formula: `cos(a,b) = dot(a,b) / (||a|| * ||b||)`. "
        "Possible directions: DTW for time alignment, or FAISS to speed things up with approximate nearest neighbours."
    ),
    (
        "How does the agent layer handle failures and retries?",
        "Right now it's a simple linear pipeline with no retry logic — this is a known limit. "
        "The production-grade approach: wrap each step as a Task, retry with exponential backoff on failure, and alert past a threshold."
    ),
    (
        "If you had three months, what would you improve first?",
        "Streaming inference > noise robustness > large-vocabulary language-model fusion. "
        "Streaming is the biggest engineering gap, noise robustness has the most direct impact in real use, "
        "and language-model (shallow) fusion can bring WER down quickly."
    ),
]

print("── Common Q&A quick reference ──\n")
for i, (q, a) in enumerate(qa, 1):
    print(f"Q{i:02d}: {q}")
    print(f"  A:  {a}")
    print()

assert len(qa) == 10
print("✅ All 10 questions prepared")


## 5. Your GitHub profile: the first impression happens in the repository

When someone (an interviewer, a potential collaborator, or even yourself a few months on) looks at your project, the first thing they usually do is open GitHub.
This checklist helps you make a good impression within 30 seconds:

- **Pinned repository**: pin `aurora` and make the description spell out "what + why from scratch"
- **README first screen**: understandable in 30 seconds, with one command you can run
- **Commit messages**: each one in plain language, e.g. `feat(fft): implement Cooley-Tukey butterfly`
- **Language stats**: mostly Python (Aurora gets this naturally)

Recommended first-screen structure for the `aurora` README:
```
# Aurora · audio AI from scratch
> FFT / STFT / Mel / MFCC / CTC — no API wrappers, pure algorithms

## Quick start
\`\`\`bash
make install && python scripts/demo_audio.py
\`\`\`
```

In [ ]:
readme_checklist = {
    "One-sentence intro (what + why from scratch)": None,  # True/False
    "A runnable quick-start command":           None,
    "Core capability table (with error/accuracy)": None,
    "At least 1 screenshot or terminal output":  None,
    "Test badge or CI status":                   None,
    "License declared":                          None,
}
done = sum(1 for v in readme_checklist.values() if v is True)
print(f"README self-check: {done}/{len(readme_checklist)}")
for k, v in readme_checklist.items():
    icon = "✅" if v is True else ("❌" if v is False else "⬜")
    print(f"  {icon} {k}")

## 6. A technical blog: one post > ten commits

Aurora is a natural fit for technical blogging — every module built from scratch is a ready-made story:
why you wrote it by hand, what pitfalls you hit, and how far your result is from the reference implementation.

| Post type | Example title |
|---|---|
| Derivation notes | "The FFT butterfly: the full derivation from the DFT definition to O(N log N)" |
| Implementation comparison | "Hand-written MFCC vs librosa: where the error comes from, and why it's designed this way" |
| Pitfall log | "Three boundary conditions in the CTC forward algorithm — two days of debugging to find them" |
| Paper notes | "Whisper, read closely: how weakly supervised data trains a strong ASR model" |

Where to publish: dev.to (developer audience) / Medium (general tech readers) / GitHub Pages (full control)

In [ ]:
first_blog = {
    "title":         None,   # What do you want your first post to be?
    "platform":      None,   # Where will you publish it?
    "core_message":  None,   # What will readers learn? (one sentence)
    "deadline":      None,   # a specific date, e.g. '2026-07-15'
}
if all(v is not None for v in first_blog.values()):
    print("Blog plan ready:")
    for k, v in first_blog.items():
        print(f"  {k}: {v}")
else:
    empty = [k for k, v in first_blog.items() if v is None]
    print(f"Still to fill in: {len(empty)} item(s): {empty}")

## Wrap-up

This lesson has no new coding task; its core output is **three deliverables**: the evidence checklist report, the WWHRL introduction, and the list of achievement bullets.
The whole Aurora system rests on `src/aurora/audio/` as its foundation and `tests/` as its verification layer, forming a complete evidence chain from the FFT to the agent.
Real competitiveness lies not in how high the metrics are, but in being able to unpack Why / How / Limits layer by layer — whether you're facing an interviewer, a colleague or your future self, those three words are more convincing than any WER number.
Next steps: record the demo screencast, update the M6 blog post in `docs/current/blog/`, and package Aurora v1 as a shareable GitHub Release.


---

→ **Next lesson** · [L95 · Reading research papers](L95_research_papers.ipynb)

> Next lesson covers **reading research papers**: the three-pass method, paper structure and writing, the submission process and research collaboration.